# MMseqs2 CTS Demo

Runs MMseqs2 `easy-cluster` on CDM genome files via the CDM Task Service.

- **Image:** `ghcr.io/kbaseincubator/cdm_mmseqs2:0.1.0`
- **Mode:** `easy-cluster`. All-vs-all clustering, no reference DB needed
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/mmseqs2/`

See [cdm_mmseqs2 repo](https://github.com/kbaseincubator/cdm_mmseqs2) for details.

## 1. Setup clients

In [ ]:
tscli = get_task_service_client()
mincli = get_minio_client()
print(tscli.whoami())

## 2. List input files

Using the shared test genome files already in MinIO with CRC64NVME checksums.

In [ ]:
objs = list(mincli.list_objects("cts", prefix="io/gavin/test_files", recursive=True))
input_files = [f"cts/{o.object_name}" for o in objs]
print(f"{len(input_files)} input files:")
for f in input_files:
    print(f)

## 3. Submit MMseqs2 easy-cluster job

In [ ]:
IMAGE = "ghcr.io/kbaseincubator/cdm_mmseqs2:0.1.0@sha256:24afa107c0dac1a6f093cc081ec19a577c4d0b156dfa8ee66bb2d3c41b97c082"
OUTPUT_DIR = "cts/io/jplfaria/output/mmseqs2/test/v1"

job = tscli.submit_job(
    IMAGE,
    input_files,
    OUTPUT_DIR,
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "easy-cluster",
        tscli.insert_files(),
        "/out/cluster_results",
        "/out/tmp",
        "--threads", "4",
    ],
    num_containers=1,
    cpus=4,
    memory="16GB",
    runtime="PT30M"
)
print("Job ID:", job.id)

## 4. Monitor job status

In [ ]:
# Re-attach if notebook restarted:
# job = tscli.get_job_by_id("PASTE_JOB_ID_HERE")

import threading, json
thread = threading.Thread(
    target=lambda: print(json.dumps(job.wait_for_completion(), indent=4)),
    daemon=True
)
thread.start()

In [ ]:
print(job.get_job_status())

## 5. Inspect output files

In [ ]:
for o in job.get_job()["outputs"]:
    print(o["crc64nvme"], o["file"])

## 6. View cluster results

`cluster_results_cluster.tsv` columns: `representative_id`, `member_id`

In [ ]:
import io, pandas as pd
cluster_obj = mincli.get_object("cts", "io/jplfaria/output/mmseqs2/test/v1/0/cluster_results_cluster.tsv")
df = pd.read_csv(io.BytesIO(cluster_obj.read()), sep="\t", header=None, names=["representative", "member"])
print(f"{len(df)} cluster memberships, {df['representative'].nunique()} clusters")
df.head(20)